# Movie Success Predictive Analytics - Exploratory Data Analysis

This notebook contains a comprehensive exploratory data analysis (EDA) for the Movie Success Predictive Analytics project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast

# Set up plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading

Let's load the dataset from the `../data/` directory. We will use the Bollywood dataset for this initial analysis.

In [ ]:
# Define path to the data directory
data_dir = '../data'

# Load a dataset
df = pd.read_csv(os.path.join(data_dir, 'bollywood.csv'))

# Display the first few rows
df.head()

## 2. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Create a 'Profit' and 'Success' metric
df['profit'] = df['revenue'] - df['budget']
df['success'] = (df['profit'] > 0).astype(int)

print(f"Number of successful movies: {df['success'].sum()} out of {len(df)}")

In [ ]:
# Parse the genres column (which is stored as a string representation of a list of dicts)
def extract_main_genre(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        if len(genres) > 0:
            return genres[0]['name']
    except:
        pass
    return 'Unknown'

df['main_genre'] = df['genres'].apply(extract_main_genre)
df[['title', 'genres', 'main_genre']].head()

## 3. Univariate Analysis

Let's look at the distribution of individual variables.

In [ ]:
# Distribution of Budgets
plt.figure(figsize=(10, 5))
sns.histplot(df['budget'], bins=20, kde=True)
plt.title('Distribution of Movie Budgets')
plt.xlabel('Budget (INR)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Distribution of Revenue
plt.figure(figsize=(10, 5))
sns.histplot(df['revenue'], bins=20, kde=True)
plt.title('Distribution of Movie Revenue')
plt.xlabel('Revenue (INR)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Distribution of IMDB Ratings (vote_average)
plt.figure(figsize=(10, 5))
sns.histplot(df['vote_average'], bins=15, kde=True)
plt.title('Distribution of IMDB Ratings')
plt.xlabel('Vote Average')
plt.ylabel('Frequency')
plt.show()

## 4. Bivariate Analysis

How do different variables interact with each other?

In [ ]:
# Budget vs Revenue
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='budget', y='revenue', hue='success', palette={0: 'red', 1: 'green'}, s=100)
plt.title('Budget vs Revenue')
plt.xlabel('Budget (INR)')
plt.ylabel('Revenue (INR)')
plt.plot([0, df['budget'].max()], [0, df['budget'].max()], 'k--', label='Break-even')
plt.legend()
plt.show()

In [ ]:
# Correlation Heatmap
numerical_cols = ['budget', 'revenue', 'popularity', 'runtime', 'vote_average', 'profit']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## 5. Categorical Insights

In [ ]:
# Average Revenue by Genre
genre_revenue = df.groupby('main_genre')['revenue'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=genre_revenue.index, y=genre_revenue.values)
plt.title('Average Revenue by Main Genre')
plt.xlabel('Genre')
plt.ylabel('Average Revenue (INR)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Top 5 Most Profitable Movies
df.sort_values(by='profit', ascending=False)[['title', 'main_genre', 'budget', 'revenue', 'profit']].head(5)